In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV ,TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


data = pd.read_csv("../data/processed_data.csv")

data = pd.get_dummies(data,columns=["Type", "Store", "IsHoliday"],drop_first=True)

X = data.drop(["Weekly_Sales","Date"], axis=1)
y = data["Weekly_Sales"]

split_index = int(len(data) * 0.8)
X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

# XGBoost Model
xg_model = XGBRegressor(objective="reg:squarederror",random_state=0,n_jobs=1)

# Parameters
params = {"n_estimators": [100, 200, 300],"max_depth": [3, 5, 7],"learning_rate": [0.01, 0.05, 0.1],"subsample": [0.8, 1.0],"colsample_bytree": [0.8, 1.0]}
tscv = TimeSeriesSplit(n_splits=3)

# Randomized Search
xgb_search = RandomizedSearchCV(estimator=xg_model,param_distributions=params,n_iter=5,cv=tscv,scoring=None,verbose=2,random_state=0)

# Train
xgb_search.fit(X_train, y_train)

print("Best Parameters:")
print(xgb_search.best_params_)

print("\nBest Score:")
print(xgb_search.best_score_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=100, subsample=0.8; total time=   4.8s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=100, subsample=0.8; total time=   9.3s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=7, n_estimators=100, subsample=0.8; total time=  14.1s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=300, subsample=0.8; total time=   8.3s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=300, subsample=0.8; total time=  23.4s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=5, n_estimators=300, subsample=0.8; total time=  36.2s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=1.0; total time=   4.9s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=1.0; total time=  10.7s
[CV] END col

In [2]:
import joblib
best_xgb = xgb_search.best_estimator_


xgb_pred = best_xgb.predict(X_test)


# Save model
joblib.dump(best_xgb, "../models/best_xgboost.pkl")

print("XGBoost model saved successfully!")
comparison = pd.DataFrame({"Actual_Sales": y_test.values,"Predicted_Sales": xgb_pred})

print(comparison.head(20))

XGBoost model saved successfully!
    Actual_Sales  Predicted_Sales
0       25890.80     21558.398438
1       91691.90     88558.632812
2       28446.31     22729.623047
3        1943.18      3692.164551
4        7563.94     14548.604492
5       10434.61      7152.139648
6       14368.12     15468.166016
7           7.97      1070.516357
8        2749.30      8759.030273
9        2019.50      4794.772461
10      27909.70     25992.501953
11       3431.50      5330.299316
12       1555.89      4797.731445
13        799.00      2714.931641
14        147.00      3026.118896
15      21711.56     23580.421875
16      40032.45     36244.164062
17       5035.63      7672.630371
18      17406.79     16409.871094
19      31151.40     17938.992188


In [3]:
xgb_mae = mean_absolute_error(y_test, xgb_pred)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

xgb_r2 = r2_score(y_test, xgb_pred)

print("XG RMSE:", xgb_rmse)
print("XG MAE:", xgb_mae)
print("XG R2 Score:", xgb_r2)

XG RMSE: 6925.335181438748
XG MAE: 4456.792768899173
XG R2 Score: 0.9004298686298166
